## Data Mining Project
(MBA DS&DA 24-26)

## Impact of RBI Rate Hikes on BSE SENSEX: A Forecasting Approach Using LSTM

Group Members
1. Anubhav Sarkar(24030242010) : Model Building & forecasting Analysis
2. Anik Basu(24030242006) : Model Building & forecasting Analysis
3. Palak Ghorpade(24030242079) : Visualisation & Interpretation 
4. Sreeja Roy(24030242066) : Visualisation & Interpretation 

## Pseudo Code for the Experiment

1. Load the dataset containing 'Date' and 'Close' values.

2. Convert 'Date' column to datetime format and set it as the index of the dataframe.

3. Create a full date range from the minimum to the maximum date.
   - Reindex the dataframe to this full date range.
   - Forward fill any missing 'Close' values.

4. Calculate a 5-day moving average on the 'Close' values.

5. Normalize the 'Close' values using MinMaxScaler.

6. Define a function to create input-output sequences from the normalized data.
   - Input: Sequence length (50)
   - Output: X, y pairs

7. Create sequences for LSTM using the scaled 'Close' values.

8. Split the sequences into training (80%) and testing (20%) datasets.
   - Reshape data into (samples, timesteps, features) format for LSTM.

9. Build an LSTM model with two layers of LSTM and one Dense output layer.

10. Compile the LSTM model using the Adam optimizer and mean squared error loss function.

11. Train the model with the training data.

12. Make predictions on the test data using the trained LSTM model.

13. Inverse scale the predictions and actual test data back to the original 'Close' values.

14. Calculate evaluation metrics:
    - Mean Absolute Percentage Error (MAPE)
    - Root Mean Squared Error (RMSE)

15. Fit a linear regression model to the 'Date' and 'Close' data to compute the linear trend.

16. Prepare visualization:
    - Plot the original 'Close' values over the full dataset.
    - Plot the LSTM forecasted 'Close' values.
    - Plot the 5-day moving average.
    - Plot the linear trend line.
    - Add vertical lines for key RBI rate increase events (e.g., June 6, 2018, and August 1, 2018).

17. Display the full plot with original data, LSTM predictions, moving average, and trend line.

18. Zoom in on the prediction period:
    - Plot only the test data section showing the LSTM predictions and actual 'Close' values.
    - Overlay LSTM predictions and actual values.
    - Display the zoomed-in plot.


In [141]:
import pandas as pd

In [142]:
test= pd.read_csv("E:/DSDA/DMsensex.csv")
test=test.drop("Unnamed: 0", axis=1)
print(test)

            Date       Open       High        Low      Close
0      1/01/2014  21,222.19  21,244.35  21,133.82  21,140.48
1      2/01/2014  21,179.91  21,331.32  20,846.67  20,888.33
2      3/01/2014  20,819.58  20,885.18  20,731.33  20,851.33
3      6/01/2014  20,913.79  20,913.79  20,721.98  20,787.30
4      7/01/2014  20,845.77  20,890.48  20,637.18  20,693.24
...          ...        ...        ...        ...        ...
1727  25/05/2023  61,706.13  61,934.01  61,484.66  61,872.62
1728  26/05/2023  61,985.36  62,529.83  61,911.61  62,501.69
1729  29/05/2023  62,801.54  63,026.00  62,801.54  62,846.38
1730  30/05/2023  62,839.85  63,036.12  62,737.40  62,969.13
1731  31/05/2023  62,839.97  62,876.77  62,401.02  62,622.24

[1732 rows x 5 columns]


In [143]:
testcols = ['Open', 'High', 'Low', 'Close']

# Convert columns to numeric
for col in testcols:
    if test[col].dtype == 'object':  # Check if the column is of type object (usually strings)
        test[col] = pd.to_numeric(test[col].str.replace(',', ''), errors='coerce')
    else:
        test[col] = pd.to_numeric(test[col], errors='coerce')

# Display the updated DataFrame
test.head()

,Date,Open,High,Low,Close
0,1/01/2014,21222.19,21244.35,21133.82,21140.48
1,2/01/2014,21179.91,21331.32,20846.67,20888.33
2,3/01/2014,20819.58,20885.18,20731.33,20851.33
3,6/01/2014,20913.79,20913.79,20721.98,20787.30
4,7/01/2014,20845.77,20890.48,20637.18,20693.24


## 2015 (RBI Rate Cut Cycle Begins)
1. January 15, 2015: Cut by 25 bps to 7.75%.
2. March 4, 2015: Cut by 25 bps to 7.50%.
3. June 2, 2015: Cut by 25 bps to 7.25%.
4. September 29, 2015: Cut by 50 bps to 6.75%

Event Dates 

In [144]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

# Key dates for interest rate cuts
jan_15 = pd.to_datetime('15/01/2015', format='%d/%m/%Y')
mar_4 = pd.to_datetime('04/03/2015', format='%d/%m/%Y')
jun_2 = pd.to_datetime('02/06/2015', format='%d/%m/%Y')
sep_29 = pd.to_datetime('29/09/2015', format='%d/%m/%Y')

# Defining the time window around these events
start_date = jan_15 - pd.Timedelta(days=70)
end_date = sep_29 + pd.Timedelta(days=70)

# Filter the dataset within this time window
filtered_data_events = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)]


Filtered data according to dates

In [145]:
filtered_data = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)].copy()
filtered_data['Date'] = pd.to_datetime(filtered_data['Date'], format='%d/%m/%Y')
print(filtered_data)

          Date      Open      High       Low     Close
65  2014-11-07  27902.71  27980.93  27739.56  27868.63
66  2014-11-10  27919.45  28027.96  27764.75  27874.73
67  2014-11-11  27911.25  27996.92  27790.40  27910.06
68  2014-11-12  27958.64  28126.48  27958.64  28008.90
69  2014-11-13  28048.56  28098.74  27822.70  27940.64
..         ...       ...       ...       ...       ...
330 2015-12-02  26239.39  26256.42  26041.68  26117.85
331 2015-12-03  26123.86  26123.86  25857.35  25886.62
332 2015-12-04  25810.06  25810.06  25623.71  25638.11
333 2015-12-07  25746.03  25785.53  25477.69  25530.11
334 2015-12-08  25488.42  25542.47  25256.79  25310.33

[270 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [146]:
filtered_data['Date'] = pd.to_datetime(filtered_data['Date'])
filtered_data.set_index('Date', inplace=True)

full_range = pd.date_range(start=filtered_data.index.min(), end=filtered_data.index.max())

filtered_data = filtered_data.reindex(full_range)

filtered_data['Close'] = filtered_data['Close'].ffill()

filtered_data.reset_index(inplace=True)
filtered_data.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data)

          Date      Open      High       Low     Close
0   2014-11-07  27902.71  27980.93  27739.56  27868.63
1   2014-11-08       NaN       NaN       NaN  27868.63
2   2014-11-09       NaN       NaN       NaN  27868.63
3   2014-11-10  27919.45  28027.96  27764.75  27874.73
4   2014-11-11  27911.25  27996.92  27790.40  27910.06
..         ...       ...       ...       ...       ...
392 2015-12-04  25810.06  25810.06  25623.71  25638.11
393 2015-12-05       NaN       NaN       NaN  25638.11
394 2015-12-06       NaN       NaN       NaN  25638.11
395 2015-12-07  25746.03  25785.53  25477.69  25530.11
396 2015-12-08  25488.42  25542.47  25256.79  25310.33

[397 rows x 5 columns]


Required Libraries

In [147]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
import plotly.graph_objs as go
from sklearn.linear_model import LinearRegression

In [148]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

Scale Data 

In [149]:

test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data['Date'] = pd.to_datetime(filtered_data['Date'])

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(filtered_data[['Close']].values)


Create sequences for LSTM input

In [150]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length = 50
X, y = create_sequences(scaled_data, seq_length)


Split the data into training and test sets & Build and compile the LSTM model

In [151]:
split_index = int(0.8 * len(X))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))


model = Sequential()
model.add(Input(shape=(seq_length, 1)))
model.add(LSTM(50, return_sequences=True))
model.add(LSTM(50))
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

Train the model 


In [152]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Compute evaluation metrics
mape = mean_absolute_percentage_error(y_test_actual, predictions) * 100
rmse_value = rmse(y_test_actual, predictions)

Epoch 1/20


9/9 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.1791
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0344
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0240
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0186
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0168
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0136
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0130
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0134
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0131
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0113
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0108
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0099
Epoch 13/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0110
Epoch 14/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0100
Epoch 15/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 0.0094
Epoch 16/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/

Make predictions and inverse scaling

In [153]:
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Compute evaluation metrics
mape = mean_absolute_percentage_error(y_test_actual, predictions) * 100
rmse_value = rmse(y_test_actual, predictions)
print(mape)
print(rmse_value)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
1.0131524607941205
334.1135846998839


Moving average and linear trend line

In [154]:
# Moving Average (5 Days)
filtered_data['Moving_Average'] = filtered_data['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear = filtered_data['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model = LinearRegression()
linear_model.fit(X_linear, filtered_data['Close'])
filtered_data['Linear_Trend'] = linear_model.predict(X_linear)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [155]:
# Plot the results for the first set of interest rate cut events

# Original Close Values
trace_original = go.Scatter(
    x=filtered_data['Date'], 
    y=filtered_data['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm = go.Scatter(
    x=filtered_data['Date'][split_index+seq_length:], 
    y=predictions.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg = go.Scatter(
    x=filtered_data['Date'], 
    y=filtered_data['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend = go.Scatter(
    x=filtered_data['Date'], 
    y=filtered_data['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for interest rate cuts (Updated for all events)
vertical_lines = [
    go.Scatter(
        x=[pd.to_datetime('2015-01-15'), pd.to_datetime('2015-01-15')],
        y=[filtered_data['Close'].min(), filtered_data['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Jan 15, 2015: Cut to 7.75%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2015-03-04'), pd.to_datetime('2015-03-04')],
        y=[filtered_data['Close'].min(), filtered_data['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Mar 4, 2015: Cut to 7.50%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2015-06-02'), pd.to_datetime('2015-06-02')],
        y=[filtered_data['Close'].min(), filtered_data['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Jun 2, 2015: Cut to 7.25%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2015-09-29'), pd.to_datetime('2015-09-29')],
        y=[filtered_data['Close'].min(), filtered_data['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Sep 29, 2015: Cut to 6.75%'
    )
]

# Combine all traces for the first graph
layout = go.Layout(
    title=f"Entire Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape:.2f}%, RMSE: {rmse_value:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig = go.Figure(data=[trace_original, trace_lstm, trace_moving_avg, trace_linear_trend] + vertical_lines, layout=layout)
fig.show()


Graph: Show zoomed-in view starting from predictions

In [156]:
# Second graph: Zoom in on prediction period only
trace_original_zoomed = go.Scatter(
    x=filtered_data['Date'][split_index+seq_length:], 
    y=y_test_actual.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed = go.Scatter(
    x=filtered_data['Date'][split_index+seq_length:], 
    y=predictions.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape:.2f}%, RMSE: {rmse_value:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed = go.Figure(data=[trace_original_zoomed, trace_lstm_zoomed], layout=layout_zoomed)
fig_zoomed.show()


## 2016 (Further RBI cuts)
1. April 5, 2016: Cut by 25 bps to 6.50%.
2. October 4, 2016: Cut by 25 bps to 6.25%.


Event Dates

In [157]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

apr_5 = pd.to_datetime('05/04/2016', format='%d/%m/%Y')
oct_4 = pd.to_datetime('04/10/2016', format='%d/%m/%Y')

start_date = apr_5 - pd.Timedelta(days=70)
end_date = oct_4 + pd.Timedelta(days=70)

filtered_data_event2 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)]


Filtered data according to dates

In [158]:
filtered_data_event2 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)].copy()
filtered_data_event2['Date'] = pd.to_datetime(filtered_data_event2['Date'], format='%d/%m/%Y')

print(filtered_data_event2)


          Date      Open      High       Low     Close
368 2016-01-27  24643.13  24645.70  24458.13  24492.39
369 2016-01-28  24481.86  24587.20  24400.52  24469.57
370 2016-01-29  24347.31  24911.90  24340.06  24870.69
371 2016-02-01  24982.22  25002.32  24788.58  24824.83
372 2016-02-02  24868.21  24928.75  24460.53  24539.00
..         ...       ...       ...       ...       ...
580 2016-12-07  26456.21  26540.83  26164.82  26236.87
581 2016-12-08  26366.52  26733.87  26357.35  26694.28
582 2016-12-09  26787.14  26803.76  26707.81  26747.18
583 2016-12-12  26725.31  26725.31  26468.59  26515.24
584 2016-12-13  26607.65  26724.97  26494.23  26697.82

[217 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [159]:
filtered_data_event2['Date'] = pd.to_datetime(filtered_data_event2['Date'])
filtered_data_event2.set_index('Date', inplace=True)

full_range_event2 = pd.date_range(start=filtered_data_event2.index.min(), end=filtered_data_event2.index.max())

filtered_data_event2 = filtered_data_event2.reindex(full_range_event2)

filtered_data_event2['Close'] = filtered_data_event2['Close'].ffill()

filtered_data_event2.reset_index(inplace=True)
filtered_data_event2.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data_event2)


          Date      Open      High       Low     Close
0   2016-01-27  24643.13  24645.70  24458.13  24492.39
1   2016-01-28  24481.86  24587.20  24400.52  24469.57
2   2016-01-29  24347.31  24911.90  24340.06  24870.69
3   2016-01-30       NaN       NaN       NaN  24870.69
4   2016-01-31       NaN       NaN       NaN  24870.69
..         ...       ...       ...       ...       ...
317 2016-12-09  26787.14  26803.76  26707.81  26747.18
318 2016-12-10       NaN       NaN       NaN  26747.18
319 2016-12-11       NaN       NaN       NaN  26747.18
320 2016-12-12  26725.31  26725.31  26468.59  26515.24
321 2016-12-13  26607.65  26724.97  26494.23  26697.82

[322 rows x 5 columns]


Scale Data

In [160]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data_event2['Date'] = pd.to_datetime(filtered_data_event2['Date'])

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data_event2 = scaler.fit_transform(filtered_data_event2[['Close']].values)


Create sequences for LSTM input

In [161]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length = 50
X_event2, y_event2 = create_sequences(scaled_data_event2, seq_length)


Split the data into training and test sets & Build and compile the LSTM model

In [162]:
split_index_event2 = int(0.8 * len(X_event2))
X_train_event2, X_test_event2 = X_event2[:split_index_event2], X_event2[split_index_event2:]
y_train_event2, y_test_event2 = y_event2[:split_index_event2], y_event2[split_index_event2:]

# Reshaping the input data
X_train_event2 = np.reshape(X_train_event2, (X_train_event2.shape[0], X_train_event2.shape[1], 1))
X_test_event2 = np.reshape(X_test_event2, (X_test_event2.shape[0], X_test_event2.shape[1], 1))

# Building the LSTM model
model_event2 = Sequential()
model_event2.add(Input(shape=(seq_length, 1)))
model_event2.add(LSTM(50, return_sequences=True))
model_event2.add(LSTM(50))
model_event2.add(Dense(1))

# Compile the model
model_event2.compile(optimizer='adam', loss='mean_squared_error')


Train the Model

In [163]:
# Train the model
history_event2 = model_event2.fit(X_train_event2, y_train_event2, epochs=20, batch_size=32, verbose=1)


Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.2472
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0335
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0092
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0091
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0068
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0037
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0042
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0036
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0039
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0038
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0035
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0031
Epoch 13/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0031
Epoch 14/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0031
Epoch 15/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0030
Epoch 16/20
7/7 ━━━━━━━━━━━━━━━━━━

Make Predictions and Inverse scaling

In [164]:
# Make predictions
predictions_event2 = model_event2.predict(X_test_event2)

# Inverse scaling for predictions
predictions_event2 = scaler.inverse_transform(predictions_event2)

# Inverse scale the true test values as well
y_test_actual_event2 = scaler.inverse_transform(y_test_event2.reshape(-1, 1))

# Compute evaluation metrics
mape_event2 = mean_absolute_percentage_error(y_test_actual_event2, predictions_event2) * 100
rmse_value_event2 = rmse(y_test_actual_event2, predictions_event2)

# Print evaluation metrics
print(mape_event2)
print(rmse_value_event2)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
1.3819916768402134
484.34581565796015


Moving Average & Liner Trend Line

In [165]:
# Moving Average (5 Days)
filtered_data_event2['Moving_Average'] = filtered_data_event2['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear_event2 = filtered_data_event2['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model_event2 = LinearRegression()
linear_model_event2.fit(X_linear_event2, filtered_data_event2['Close'])
filtered_data_event2['Linear_Trend'] = linear_model_event2.predict(X_linear_event2)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [166]:
# Original Close Values
trace_original_event2 = go.Scatter(
    x=filtered_data_event2['Date'], 
    y=filtered_data_event2['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm_event2 = go.Scatter(
    x=filtered_data_event2['Date'][split_index_event2 + seq_length:], 
    y=predictions_event2.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg_event2 = go.Scatter(
    x=filtered_data_event2['Date'], 
    y=filtered_data_event2['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend_event2 = go.Scatter(
    x=filtered_data_event2['Date'], 
    y=filtered_data_event2['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for interest rate cuts (Updated for the new events)
vertical_lines_event2 = [
    go.Scatter(
        x=[pd.to_datetime('2016-04-05'), pd.to_datetime('2016-04-05')],
        y=[filtered_data_event2['Close'].min(), filtered_data_event2['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Apr 5, 2016: Cut to 6.50%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2016-10-04'), pd.to_datetime('2016-10-04')],
        y=[filtered_data_event2['Close'].min(), filtered_data_event2['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Oct 4, 2016: Cut to 6.25%'
    )
]

# Combine all traces for the specific graph
layout_event2 = go.Layout(
    title=f"Entire Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape_event2:.2f}%, RMSE: {rmse_value_event2:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_event2 = go.Figure(data=[trace_original_event2, trace_lstm_event2, trace_moving_avg_event2, trace_linear_trend_event2] + vertical_lines_event2, layout=layout_event2)
fig_event2.show()


Graph: Show zoomed-in view starting from predictions

In [167]:
# Second graph: Zoom in on prediction period only
trace_original_zoomed_event2 = go.Scatter(
    x=filtered_data_event2['Date'][split_index_event2 + seq_length:], 
    y=y_test_actual_event2.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed_event2 = go.Scatter(
    x=filtered_data_event2['Date'][split_index_event2 + seq_length:], 
    y=predictions_event2.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed_event2 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape_event2:.2f}%, RMSE: {rmse_value_event2:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed_event2 = go.Figure(data=[trace_original_zoomed_event2, trace_lstm_zoomed_event2], layout=layout_zoomed_event2)
fig_zoomed_event2.show()


## 2017
1. August 2, 2017: Cut by 25 bps to 6.00%

Event dates


In [168]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

aug_2 = pd.to_datetime('02/08/2017', format='%d/%m/%Y')

start_date = aug_2 - pd.Timedelta(days=70)
end_date = aug_2 + pd.Timedelta(days=70)

filtered_data_event3 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)]

Filtered data according to dates

In [169]:
filtered_data_event3 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)].copy()
filtered_data_event3['Date'] = pd.to_datetime(filtered_data_event3['Date'], format='%d/%m/%Y')

print(filtered_data_event3)


          Date      Open      High       Low     Close
614 2017-05-24  30446.77  30534.15  30247.60  30301.64
615 2017-05-25  30374.81  30793.43  30352.26  30750.03
616 2017-05-26  30765.77  31074.07  30745.57  31028.21
617 2017-05-29  30944.38  31214.39  30869.90  31109.28
618 2017-05-30  31111.73  31220.38  31064.04  31159.40
..         ...       ...       ...       ...       ...
706 2017-10-05  31725.85  31772.41  31562.25  31592.03
707 2017-10-06  31633.34  31844.28  31632.81  31814.22
708 2017-10-09  31862.20  31935.63  31781.75  31846.89
709 2017-10-10  31910.82  31994.77  31896.90  31924.41
710 2017-10-11  31975.59  32098.46  31769.40  31833.99

[97 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [170]:
filtered_data_event3['Date'] = pd.to_datetime(filtered_data_event3['Date'])
filtered_data_event3.set_index('Date', inplace=True)

full_range_event3 = pd.date_range(start=filtered_data_event3.index.min(), end=filtered_data_event3.index.max())

filtered_data_event3 = filtered_data_event3.reindex(full_range_event3)

filtered_data_event3['Close'] = filtered_data_event3['Close'].ffill()

filtered_data_event3.reset_index(inplace=True)
filtered_data_event3.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data_event3)


          Date      Open      High       Low     Close
0   2017-05-24  30446.77  30534.15  30247.60  30301.64
1   2017-05-25  30374.81  30793.43  30352.26  30750.03
2   2017-05-26  30765.77  31074.07  30745.57  31028.21
3   2017-05-27       NaN       NaN       NaN  31028.21
4   2017-05-28       NaN       NaN       NaN  31028.21
..         ...       ...       ...       ...       ...
136 2017-10-07       NaN       NaN       NaN  31814.22
137 2017-10-08       NaN       NaN       NaN  31814.22
138 2017-10-09  31862.20  31935.63  31781.75  31846.89
139 2017-10-10  31910.82  31994.77  31896.90  31924.41
140 2017-10-11  31975.59  32098.46  31769.40  31833.99

[141 rows x 5 columns]


Scale data

In [171]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data_event3['Date'] = pd.to_datetime(filtered_data_event3['Date'])

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data_event3 = scaler.fit_transform(filtered_data_event3[['Close']].values)


Create sequences for LSTM input

In [172]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length = 50
X_event3, y_event3 = create_sequences(scaled_data_event3, seq_length)


Split the data into training and test sets & Build and compile the LSTM model

In [173]:
split_index_event3 = int(0.8 * len(X_event3))
X_train_event3, X_test_event3 = X_event3[:split_index_event3], X_event3[split_index_event3:]
y_train_event3, y_test_event3 = y_event3[:split_index_event3], y_event3[split_index_event3:]

# Reshaping the input data
X_train_event3 = np.reshape(X_train_event3, (X_train_event3.shape[0], X_train_event3.shape[1], 1))
X_test_event3 = np.reshape(X_test_event3, (X_test_event3.shape[0], X_test_event3.shape[1], 1))

# Building the LSTM model
model_event3 = Sequential()
model_event3.add(Input(shape=(seq_length, 1)))
model_event3.add(LSTM(50, return_sequences=True))
model_event3.add(LSTM(50))
model_event3.add(Dense(1))

# Compile the model
model_event3.compile(optimizer='adam', loss='mean_squared_error')


Train the model

In [174]:
# Train the model
history_event3 = model_event3.fit(X_train_event3, y_train_event3, epochs=20, batch_size=32, verbose=1)


Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.2885
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0845
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0553
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0513
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0294
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0332
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0342
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0257
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0230
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0267
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0249
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0187
Epoch 13/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0220
Epoch 14/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0214
Epoch 15/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0221
Epoch 16/20
3/3 ━━━━━━━━━━━━━━━━━━

Make Prediction and Inverse scaling

In [175]:
predictions_event3 = model_event3.predict(X_test_event3)

# Inverse scaling for predictions
predictions_event3 = scaler.inverse_transform(predictions_event3)

# Inverse scale the true test values 
y_test_actual_event3 = scaler.inverse_transform(y_test_event3.reshape(-1, 1))

# Compute evaluation metrics
mape_event3 = mean_absolute_percentage_error(y_test_actual_event3, predictions_event3) * 100
rmse_value_event3 = rmse(y_test_actual_event3, predictions_event3)

print(mape_event3)
print(rmse_value_event3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
1.0713975996337037
407.6002728012823


Moving Average & Linear Trend line

In [176]:
# Moving Average (5 Days)
filtered_data_event3['Moving_Average'] = filtered_data_event3['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear_event3 = filtered_data_event3['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model_event3 = LinearRegression()
linear_model_event3.fit(X_linear_event3, filtered_data_event3['Close'])
filtered_data_event3['Linear_Trend'] = linear_model_event3.predict(X_linear_event3)


Graph 
1. Show entire dataset (original + prediction + moving average + trend)
2. Show zoomed-in view starting from predictions

In [177]:
# Original Close Values
trace_original_event3 = go.Scatter(
    x=filtered_data_event3['Date'], 
    y=filtered_data_event3['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm_event3 = go.Scatter(
    x=filtered_data_event3['Date'][split_index_event3 + seq_length:], 
    y=predictions_event3.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg_event3 = go.Scatter(
    x=filtered_data_event3['Date'], 
    y=filtered_data_event3['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend_event3 = go.Scatter(
    x=filtered_data_event3['Date'], 
    y=filtered_data_event3['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for interest rate cuts (Updated for the new events)
vertical_lines_event3 = [
    go.Scatter(
        x=[pd.to_datetime('2017-08-02'), pd.to_datetime('2017-08-02')],
        y=[filtered_data_event3['Close'].min(), filtered_data_event3['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Aug 2, 2017: Cut to 6.00%'
    )
]

# Combine all traces for the specific graph
layout_event3 = go.Layout(
    title=f"Entire Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape_event3:.2f}%, RMSE: {rmse_value_event3:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_event3 = go.Figure(data=[trace_original_event3, trace_lstm_event3, trace_moving_avg_event3, trace_linear_trend_event3] + vertical_lines_event3, layout=layout_event3)
fig_event3.show()

# Second graph: Zoom in on prediction period only
trace_original_zoomed_event3 = go.Scatter(
    x=filtered_data_event3['Date'][split_index_event3 + seq_length:], 
    y=y_test_actual_event3.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed_event3 = go.Scatter(
    x=filtered_data_event3['Date'][split_index_event3 + seq_length:], 
    y=predictions_event3.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed_event3 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape_event3:.2f}%, RMSE: {rmse_value_event3:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed_event3 = go.Figure(data=[trace_original_zoomed_event3, trace_lstm_zoomed_event3], layout=layout_zoomed_event3)
fig_zoomed_event3.show()


## 2018 (RBI rate increased)
1. June 6, 2018: Rate increased by 25 bps to 6.25%.
2. August 1, 2018: Rate increased by 25 bps to 6.50%.


Event Dates

In [178]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

jun_6_2018 = pd.to_datetime('06/06/2018', format='%d/%m/%Y')
aug_1_2018 = pd.to_datetime('01/08/2018', format='%d/%m/%Y')

start_date = jun_6_2018 - pd.Timedelta(days=70)
end_date = aug_1_2018 + pd.Timedelta(days=70)

filtered_data_event_2018 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)]


Filtered data according to dates

In [179]:

filtered_data_event_2018 = test[(test['Date'] >= start_date) & (test['Date'] <= end_date)].copy()

filtered_data_event_2018['Date'] = pd.to_datetime(filtered_data_event_2018['Date'], format='%d/%m/%Y')

print(filtered_data_event_2018)


          Date      Open      High       Low     Close
784 2018-03-28  33098.09  33104.11  32917.66  32968.68
785 2018-04-02  33030.87  33289.34  32997.88  33255.36
786 2018-04-03  33197.42  33402.94  33153.83  33370.63
787 2018-04-04  33437.52  33505.53  32972.56  33019.07
788 2018-04-05  33289.96  33637.46  33267.86  33596.80
..         ...       ...       ...       ...       ...
912 2018-10-04  35820.53  35820.53  35022.12  35169.16
913 2018-10-05  35097.99  35118.54  34202.22  34376.99
914 2018-10-08  34412.36  34636.43  33974.66  34474.38
915 2018-10-09  34651.82  34711.68  34233.50  34299.47
916 2018-10-10  34493.21  34858.35  34346.50  34760.89

[133 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [180]:

filtered_data_event_2018['Date'] = pd.to_datetime(filtered_data_event_2018['Date'])

filtered_data_event_2018.set_index('Date', inplace=True)

full_range_event_2018 = pd.date_range(start=filtered_data_event_2018.index.min(), end=filtered_data_event_2018.index.max())

filtered_data_event_2018 = filtered_data_event_2018.reindex(full_range_event_2018)

filtered_data_event_2018['Close'] = filtered_data_event_2018['Close'].ffill()

filtered_data_event_2018.reset_index(inplace=True)

filtered_data_event_2018.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data_event_2018)


          Date      Open      High       Low     Close
0   2018-03-28  33098.09  33104.11  32917.66  32968.68
1   2018-03-29       NaN       NaN       NaN  32968.68
2   2018-03-30       NaN       NaN       NaN  32968.68
3   2018-03-31       NaN       NaN       NaN  32968.68
4   2018-04-01       NaN       NaN       NaN  32968.68
..         ...       ...       ...       ...       ...
192 2018-10-06       NaN       NaN       NaN  34376.99
193 2018-10-07       NaN       NaN       NaN  34376.99
194 2018-10-08  34412.36  34636.43  33974.66  34474.38
195 2018-10-09  34651.82  34711.68  34233.50  34299.47
196 2018-10-10  34493.21  34858.35  34346.50  34760.89

[197 rows x 5 columns]


Scale Data

In [181]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data_event_2018['Date'] = pd.to_datetime(filtered_data_event_2018['Date'])

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data_event_2018 = scaler.fit_transform(filtered_data_event_2018[['Close']].values)


Create sequences for LSTM input

In [182]:
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length = 40

# Create sequences for the 2018 event data
X_event_2018, y_event_2018 = create_sequences(scaled_data_event_2018, seq_length)


Split the data into training and test sets & Build and compile the LSTM model

In [183]:
# Splitting the data into training and testing sets
split_index_event_2018 = int(0.8 * len(X_event_2018))
X_train_event_2018, X_test_event_2018 = X_event_2018[:split_index_event_2018], X_event_2018[split_index_event_2018:]
y_train_event_2018, y_test_event_2018 = y_event_2018[:split_index_event_2018], y_event_2018[split_index_event_2018:]

# Reshaping the input data for the LSTM model
X_train_event_2018 = np.reshape(X_train_event_2018, (X_train_event_2018.shape[0], X_train_event_2018.shape[1], 1))
X_test_event_2018 = np.reshape(X_test_event_2018, (X_test_event_2018.shape[0], X_test_event_2018.shape[1], 1))

# Building the LSTM model
model_event_2018 = Sequential()
model_event_2018.add(Input(shape=(seq_length, 1)))
model_event_2018.add(LSTM(50, return_sequences=True))
model_event_2018.add(LSTM(50))
model_event_2018.add(Dense(1))

# Compile the model
model_event_2018.compile(optimizer='adam', loss='mean_squared_error')


Train the Model

In [184]:
history_event_2018 = model_event_2018.fit(X_train_event_2018, y_train_event_2018, epochs=25, batch_size=32, verbose=1)


Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.2199
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0279
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0361
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0062
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0146
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0107
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0051
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0074
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0059
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0048
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0049
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0036
Epoch 13/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0037
Epoch 14/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0033
Epoch 15/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0038
Epoch 16/25
4/4 ━━━━━━━━━━━━━━━━━━

Make Predictions and Inverse scaling

In [185]:
# Making predictions using the LSTM model
predictions_event_2018 = model_event_2018.predict(X_test_event_2018)

# Inverse scaling for predictions
predictions_event_2018 = scaler.inverse_transform(predictions_event_2018)

# Inverse scale the true test values as well
y_test_actual_event_2018 = scaler.inverse_transform(y_test_event_2018.reshape(-1, 1))

# Compute evaluation metrics
mape_event_2018 = mean_absolute_percentage_error(y_test_actual_event_2018, predictions_event_2018) * 100
rmse_value_event_2018 = rmse(y_test_actual_event_2018, predictions_event_2018)

# Print evaluation metrics
print(mape_event_2018)
print(rmse_value_event_2018)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
3.3571269636548093
1321.3277816353516


Moving Average & Liner Trend Line

In [186]:
# Calculate the moving average with a window of 5 days
filtered_data_event_2018['Moving_Average'] = filtered_data_event_2018['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear_event_2018 = filtered_data_event_2018['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model_event_2018 = LinearRegression()
linear_model_event_2018.fit(X_linear_event_2018, filtered_data_event_2018['Close'])
filtered_data_event_2018['Linear_Trend'] = linear_model_event_2018.predict(X_linear_event_2018)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [187]:
import plotly.graph_objects as go

# Original Close Values
trace_original_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'], 
    y=filtered_data_event_2018['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'][split_index_event_2018 + seq_length:], 
    y=predictions_event_2018.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'], 
    y=filtered_data_event_2018['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'], 
    y=filtered_data_event_2018['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for RBI rate increase events
vertical_lines_event_2018 = [
    go.Scatter(
        x=[pd.to_datetime('2018-06-06'), pd.to_datetime('2018-06-06')],
        y=[filtered_data_event_2018['Close'].min(), filtered_data_event_2018['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='June 6, 2018: Rate Increase to 6.25%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2018-08-01'), pd.to_datetime('2018-08-01')],
        y=[filtered_data_event_2018['Close'].min(), filtered_data_event_2018['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='August 1, 2018: Rate Increase to 6.50%'
    )
]

# Combine all traces for the specific graph
layout_event_2018 = go.Layout(
    title=f"Entire Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape_event_2018:.2f}%, RMSE: {rmse_value_event_2018:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_event_2018 = go.Figure(data=[trace_original_event_2018, trace_lstm_event_2018, trace_moving_avg_event_2018, trace_linear_trend_event_2018] + vertical_lines_event_2018, layout=layout_event_2018)
fig_event_2018.show()


Graph: Show zoomed-in view starting from predictions

In [188]:
import plotly.graph_objects as go

# Zoomed-in graph: Original Close Values and LSTM Predictions
trace_original_zoomed_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'][split_index_event_2018 + seq_length:], 
    y=y_test_actual_event_2018.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed_event_2018 = go.Scatter(
    x=filtered_data_event_2018['Date'][split_index_event_2018 + seq_length:], 
    y=predictions_event_2018.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for the zoomed-in graph
layout_zoomed_event_2018 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape_event_2018:.2f}%, RMSE: {rmse_value_event_2018:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed_event_2018 = go.Figure(data=[trace_original_zoomed_event_2018, trace_lstm_zoomed_event_2018], layout=layout_zoomed_event_2018)
fig_zoomed_event_2018.show()


## 2019 (Aggressive Rate Cut Cycle):
1. February 7, 2019: Cut by 25 bps to 6.25%.
2. April 4, 2019: Cut by 25 bps to 6.00%.
3. June 6, 2019: Cut by 25 bps to 5.75%.
4. August 7, 2019: Cut by 35 bps to 5.40%.
5. October 4, 2019: Cut by 25 bps to 5.15%. 


Event Dates 

In [189]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

# Key dates for 2019 interest rate cuts
feb_7 = pd.to_datetime('07/02/2019', format='%d/%m/%Y')
apr_4 = pd.to_datetime('04/04/2019', format='%d/%m/%Y')
jun_6 = pd.to_datetime('06/06/2019', format='%d/%m/%Y')
aug_7 = pd.to_datetime('07/08/2019', format='%d/%m/%Y')
oct_4 = pd.to_datetime('04/10/2019', format='%d/%m/%Y')

# Defining the time window around these events
start_date5 = feb_7 - pd.Timedelta(days=70)
end_date5 = oct_4 + pd.Timedelta(days=70)

# Filter the dataset within this time window
filtered_data5 = test[(test['Date'] >= start_date5) & (test['Date'] <= end_date5)]


Filtered data according to dates

In [190]:
filtered_data5 = test[(test['Date'] >= start_date5) & (test['Date'] <= end_date5)].copy()
filtered_data5['Date'] = pd.to_datetime(filtered_data5['Date'], format='%d/%m/%Y')
print(filtered_data5)


           Date      Open      High       Low     Close
949  2018-11-29  35997.29  36253.85  35946.24  36170.41
950  2018-11-30  36304.43  36389.22  36082.97  36194.30
951  2018-12-03  36396.69  36446.16  36099.68  36241.00
952  2018-12-04  36290.48  36295.84  36036.39  36134.31
953  2018-12-05  36035.65  36048.65  35777.81  35884.41
...         ...       ...       ...       ...       ...
1200 2019-12-09  40527.24  40645.63  40336.56  40487.43
1201 2019-12-10  40588.81  40588.81  40208.70  40239.88
1202 2019-12-11  40285.20  40466.13  40135.37  40412.57
1203 2019-12-12  40561.34  40712.65  40490.69  40581.71
1204 2019-12-13  40754.82  41055.80  40736.70  41009.71

[256 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [191]:
filtered_data5['Date'] = pd.to_datetime(filtered_data5['Date'])
filtered_data5.set_index('Date', inplace=True)

full_range5 = pd.date_range(start=filtered_data5.index.min(), end=filtered_data5.index.max())

filtered_data5 = filtered_data5.reindex(full_range5)

filtered_data5['Close'] = filtered_data5['Close'].ffill()

filtered_data5.reset_index(inplace=True)
filtered_data5.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data5)


          Date      Open      High       Low     Close
0   2018-11-29  35997.29  36253.85  35946.24  36170.41
1   2018-11-30  36304.43  36389.22  36082.97  36194.30
2   2018-12-01       NaN       NaN       NaN  36194.30
3   2018-12-02       NaN       NaN       NaN  36194.30
4   2018-12-03  36396.69  36446.16  36099.68  36241.00
..         ...       ...       ...       ...       ...
375 2019-12-09  40527.24  40645.63  40336.56  40487.43
376 2019-12-10  40588.81  40588.81  40208.70  40239.88
377 2019-12-11  40285.20  40466.13  40135.37  40412.57
378 2019-12-12  40561.34  40712.65  40490.69  40581.71
379 2019-12-13  40754.82  41055.80  40736.70  41009.71

[380 rows x 5 columns]


Required Libraries

In [192]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
import plotly.graph_objs as go
from sklearn.linear_model import LinearRegression

In [193]:
def rmse5(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


Scale Data 

In [194]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data5['Date'] = pd.to_datetime(filtered_data5['Date'])

scaler5 = MinMaxScaler(feature_range=(0, 1))
scaled_data5 = scaler5.fit_transform(filtered_data5[['Close']].values)


Create sequences for LSTM input

In [195]:
def create_sequences5(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length5 = 50
X5, y5 = create_sequences5(scaled_data5, seq_length5)


Split the data into training and test sets & Build and compile the LSTM model

In [196]:
split_index5 = int(0.8 * len(X5))
X_train5, X_test5 = X5[:split_index5], X5[split_index5:]
y_train5, y_test5 = y5[:split_index5], y5[split_index5:]

X_train5 = np.reshape(X_train5, (X_train5.shape[0], X_train5.shape[1], 1))
X_test5 = np.reshape(X_test5, (X_test5.shape[0], X_test5.shape[1], 1))

model5 = Sequential()
model5.add(Input(shape=(seq_length5, 1)))
model5.add(LSTM(50, return_sequences=True))
model5.add(LSTM(50))
model5.add(Dense(1))

# Compile the model
model5.compile(optimizer='adam', loss='mean_squared_error')


Train the model 


In [197]:
history5 = model5.fit(X_train5, y_train5, epochs=20, batch_size=32, verbose=1)
predictions5 = model5.predict(X_test5)
predictions5 = scaler5.inverse_transform(predictions5)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual5 = scaler5.inverse_transform(y_test5.reshape(-1, 1))

# Compute evaluation metrics
mape5 = mean_absolute_percentage_error(y_test_actual5, predictions5) * 100
rmse_value5 = rmse5(y_test_actual5, predictions5)


Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.2045
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0427
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0220
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0150
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0138
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0134
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0112
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0110
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0107
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0093
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0097
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0099
Epoch 13/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0094
Epoch 14/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0088
Epoch 15/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0086
Epoch 16/20
9/9 ━━━━━━━━━━━━━━━━━━

Make predictions and inverse scaling

In [198]:
predictions5 = model5.predict(X_test5)
predictions5 = scaler5.inverse_transform(predictions5)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual5 = scaler5.inverse_transform(y_test5.reshape(-1, 1))

# Compute evaluation metrics
mape5 = mean_absolute_percentage_error(y_test_actual5, predictions5) * 100
rmse_value5 = rmse5(y_test_actual5, predictions5)

# Print the results
print(mape5)
print(rmse_value5)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
0.6638289662184789
355.60219120696684


Moving average and linear trend line

In [199]:
# Moving Average (5 Days)
filtered_data5['Moving_Average5'] = filtered_data5['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear5 = filtered_data5['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model5 = LinearRegression()
linear_model5.fit(X_linear5, filtered_data5['Close'])
filtered_data5['Linear_Trend5'] = linear_model5.predict(X_linear5)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [200]:
# Plot the results for the 2019 interest rate cut events

# Original Close Values
trace_original5 = go.Scatter(
    x=filtered_data5['Date'], 
    y=filtered_data5['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm5 = go.Scatter(
    x=filtered_data5['Date'][split_index5+seq_length5:], 
    y=predictions5.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg5 = go.Scatter(
    x=filtered_data5['Date'], 
    y=filtered_data5['Moving_Average5'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend5 = go.Scatter(
    x=filtered_data5['Date'], 
    y=filtered_data5['Linear_Trend5'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for 2019 interest rate cuts
vertical_lines5 = [
    go.Scatter(
        x=[pd.to_datetime('2019-02-07'), pd.to_datetime('2019-02-07')],
        y=[filtered_data5['Close'].min(), filtered_data5['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Feb 7, 2019: Cut to 6.25%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2019-04-04'), pd.to_datetime('2019-04-04')],
        y=[filtered_data5['Close'].min(), filtered_data5['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Apr 4, 2019: Cut to 6.00%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2019-06-06'), pd.to_datetime('2019-06-06')],
        y=[filtered_data5['Close'].min(), filtered_data5['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Jun 6, 2019: Cut to 5.75%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2019-08-07'), pd.to_datetime('2019-08-07')],
        y=[filtered_data5['Close'].min(), filtered_data5['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Aug 7, 2019: Cut to 5.40%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2019-10-04'), pd.to_datetime('2019-10-04')],
        y=[filtered_data5['Close'].min(), filtered_data5['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Oct 4, 2019: Cut to 5.15%'
    )
]

# Combine all traces for the first graph
layout5 = go.Layout(
    title=f"2019 Rate Cuts with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape5:.2f}%, RMSE: {rmse_value5:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig5 = go.Figure(data=[trace_original5, trace_lstm5, trace_moving_avg5, trace_linear_trend5] + vertical_lines5, layout=layout5)
fig5.show()


Graph: Show zoomed-in view starting from predictions

In [201]:
# Second graph: Zoom in on prediction period only
trace_original_zoomed5 = go.Scatter(
    x=filtered_data5['Date'][split_index5+seq_length5:], 
    y=y_test_actual5.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed5 = go.Scatter(
    x=filtered_data5['Date'][split_index5+seq_length5:], 
    y=predictions5.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed5 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape5:.2f}%, RMSE: {rmse_value5:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed5 = go.Figure(data=[trace_original_zoomed5, trace_lstm_zoomed5], layout=layout_zoomed5)
fig_zoomed5.show()


## 2020 (COVID-19 Pandemic Response):
1. March 27, 2020: Cut by 75 bps to 4.40%.
2. May 22, 2020: Cut by 40 bps to 4.00%. 

Event Dates 

In [202]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

# Key dates for interest rate cuts in 2020
mar_27_2020 = pd.to_datetime('27/03/2020', format='%d/%m/%Y')
may_22_2020 = pd.to_datetime('22/05/2020', format='%d/%m/%Y')

# Defining the time window around these events
start_date6 = mar_27_2020 - pd.Timedelta(days=70)
end_date6 = may_22_2020 + pd.Timedelta(days=70)

# Filter the dataset within this time window
filtered_data6 = test[(test['Date'] >= start_date6) & (test['Date'] <= end_date6)]


Filtered data according to dates

In [203]:
filtered_data6 = test[(test['Date'] >= start_date6) & (test['Date'] <= end_date6)].copy()
filtered_data6['Date'] = pd.to_datetime(filtered_data6['Date'], format='%d/%m/%Y')
print(filtered_data6)


           Date      Open      High       Low     Close
1228 2020-01-17  41929.02  42063.93  41850.29  41945.37
1229 2020-01-20  42263.00  42273.87  41503.37  41528.91
1230 2020-01-21  41487.57  41532.59  41294.30  41323.81
1231 2020-01-22  41467.13  41532.29  41059.04  41115.38
1232 2020-01-23  41191.50  41413.96  41098.91  41386.40
...         ...       ...       ...       ...       ...
1357 2020-07-27  38275.34  38275.34  37769.44  37934.73
1358 2020-07-28  38052.18  38554.72  37998.13  38492.95
1359 2020-07-29  38427.15  38617.03  37884.41  38071.13
1360 2020-07-30  38262.83  38413.81  37678.42  37736.07
1361 2020-07-31  37847.88  37897.78  37431.68  37606.89

[134 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [204]:
filtered_data6['Date'] = pd.to_datetime(filtered_data6['Date'])
filtered_data6.set_index('Date', inplace=True)

full_range6 = pd.date_range(start=filtered_data6.index.min(), end=filtered_data6.index.max())

filtered_data6 = filtered_data6.reindex(full_range6)

filtered_data6['Close'] = filtered_data6['Close'].ffill()

filtered_data6.reset_index(inplace=True)
filtered_data6.rename(columns={'index': 'Date'}, inplace=True)

print(filtered_data6)


          Date      Open      High       Low     Close
0   2020-01-17  41929.02  42063.93  41850.29  41945.37
1   2020-01-18       NaN       NaN       NaN  41945.37
2   2020-01-19       NaN       NaN       NaN  41945.37
3   2020-01-20  42263.00  42273.87  41503.37  41528.91
4   2020-01-21  41487.57  41532.59  41294.30  41323.81
..         ...       ...       ...       ...       ...
192 2020-07-27  38275.34  38275.34  37769.44  37934.73
193 2020-07-28  38052.18  38554.72  37998.13  38492.95
194 2020-07-29  38427.15  38617.03  37884.41  38071.13
195 2020-07-30  38262.83  38413.81  37678.42  37736.07
196 2020-07-31  37847.88  37897.78  37431.68  37606.89

[197 rows x 5 columns]


Required Libraries

In [205]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
import plotly.graph_objs as go
from sklearn.linear_model import LinearRegression

In [206]:
def rmse6(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


Scale Data 

In [207]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data6['Date'] = pd.to_datetime(filtered_data6['Date'])

scaler6 = MinMaxScaler(feature_range=(0, 1))
scaled_data6 = scaler6.fit_transform(filtered_data6[['Close']].values)


Create sequences for LSTM input

In [208]:
def create_sequences6(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Hyperparameters
seq_length6 = 50
X6, y6 = create_sequences6(scaled_data6, seq_length6)


Split the data into training and test sets & Build and compile the LSTM model

In [209]:
split_index6 = int(0.8 * len(X6))
X_train6, X_test6 = X6[:split_index6], X6[split_index6:]
y_train6, y_test6 = y6[:split_index6], y6[split_index6:]

X_train6 = np.reshape(X_train6, (X_train6.shape[0], X_train6.shape[1], 1))
X_test6 = np.reshape(X_test6, (X_test6.shape[0], X_test6.shape[1], 1))

model6 = Sequential()
model6.add(Input(shape=(seq_length6, 1)))
model6.add(LSTM(50, return_sequences=True))
model6.add(LSTM(50))
model6.add(Dense(1))

# Compile the model
model6.compile(optimizer='adam', loss='mean_squared_error')


Train the model 

In [210]:
history6 = model6.fit(X_train6, y_train6, epochs=20, batch_size=32, verbose=1)
predictions6 = model6.predict(X_test6)
predictions6 = scaler6.inverse_transform(predictions6)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual6 = scaler6.inverse_transform(y_test6.reshape(-1, 1))

# Compute evaluation metrics
mape6 = mean_absolute_percentage_error(y_test_actual6, predictions6) * 100
rmse_value6 = rmse6(y_test_actual6, predictions6)



Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.2043
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0671
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0200
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0294
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0137
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0159
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0144
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0126
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0101
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0098
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0093
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0092
Epoch 13/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0081
Epoch 14/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0088
Epoch 15/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0079
Epoch 16/20
4/4 ━━━━━━━━━━━━━━━━━━

Make predictions and inverse scaling

In [211]:
predictions6 = model6.predict(X_test6)
predictions6 = scaler6.inverse_transform(predictions6)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual6 = scaler6.inverse_transform(y_test6.reshape(-1, 1))

# Compute evaluation metrics
mape6 = mean_absolute_percentage_error(y_test_actual6, predictions6) * 100
rmse_value6 = rmse6(y_test_actual6, predictions6)

print(mape6)
print(rmse_value6)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
5.598653023485248
2150.777652609615


Moving average and linear trend line

In [212]:
# Moving Average (5 Days)
filtered_data6['Moving_Average'] = filtered_data6['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear6 = filtered_data6['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model6 = LinearRegression()
linear_model6.fit(X_linear6, filtered_data6['Close'])
filtered_data6['Linear_Trend'] = linear_model6.predict(X_linear6)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [213]:
# Plot the results for the 2020 interest rate cut events

# Original Close Values
trace_original6 = go.Scatter(
    x=filtered_data6['Date'], 
    y=filtered_data6['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm6 = go.Scatter(
    x=filtered_data6['Date'][split_index6+seq_length6:], 
    y=predictions6.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg6 = go.Scatter(
    x=filtered_data6['Date'], 
    y=filtered_data6['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend6 = go.Scatter(
    x=filtered_data6['Date'], 
    y=filtered_data6['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for interest rate cuts (Updated for 2020 events)
vertical_lines6 = [
    go.Scatter(
        x=[pd.to_datetime('2020-03-27'), pd.to_datetime('2020-03-27')],
        y=[filtered_data6['Close'].min(), filtered_data6['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Mar 27, 2020: Cut to 4.40%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2020-05-22'), pd.to_datetime('2020-05-22')],
        y=[filtered_data6['Close'].min(), filtered_data6['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='May 22, 2020: Cut to 4.00%'
    )
]

# Combine all traces for the graph
layout6 = go.Layout(
    title=f"2020 Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape6:.2f}%, RMSE: {rmse_value6:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig6 = go.Figure(data=[trace_original6, trace_lstm6, trace_moving_avg6, trace_linear_trend6] + vertical_lines6, layout=layout6)
fig6.show()


Graph: Show zoomed-in view starting from predictions

In [214]:
# Second graph: Zoom in on prediction period only
trace_original_zoomed6 = go.Scatter(
    x=filtered_data6['Date'][split_index6+seq_length6:], 
    y=y_test_actual6.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

trace_lstm_zoomed6 = go.Scatter(
    x=filtered_data6['Date'][split_index6+seq_length6:], 
    y=predictions6.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed6 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape6:.2f}%, RMSE: {rmse_value6:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed6 = go.Figure(data=[trace_original_zoomed6, trace_lstm_zoomed6], layout=layout_zoomed6)
fig_zoomed6.show()


## 2022 (Rate Hikes Due to Inflation):
1. May 4, 2022: Increased by 40 bps to 4.40%.
2. June 8, 2022: Increased by 50 bps to 4.90%.
3. August 5, 2022: Increased by 50 bps to 5.40%.
4. September 30, 2022: Increased by 50 bps to 5.90%.
5. December 7, 2022: Increased by 35 bps to 6.25%.
## 2023:
1. February 8, 2023: Increased by 25 bps to 6.50%.

Event Dates 

In [215]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')

# Key dates for interest rate hikes
may_4_2022 = pd.to_datetime('04/05/2022', format='%d/%m/%Y')
jun_8_2022 = pd.to_datetime('08/06/2022', format='%d/%m/%Y')
aug_5_2022 = pd.to_datetime('05/08/2022', format='%d/%m/%Y')
sep_30_2022 = pd.to_datetime('30/09/2022', format='%d/%m/%Y')
dec_7_2022 = pd.to_datetime('07/12/2022', format='%d/%m/%Y')
feb_8_2023 = pd.to_datetime('08/02/2023', format='%d/%m/%Y')

# Defining the time window around these events
start_date7 = may_4_2022 - pd.Timedelta(days=70)
end_date7 = feb_8_2023 + pd.Timedelta(days=70)

# Filter the dataset within this time window
filtered_data7 = test[(test['Date'] >= start_date7) & (test['Date'] <= end_date7)]

# Proceed with other steps using filtered_data7, split_index7, seq_length7, etc.


Filtered data according to dates

In [216]:
# Filter the dataset within the new time window for 2022-2023 rate hikes
filtered_data7 = test[(test['Date'] >= start_date7) & (test['Date'] <= end_date7)].copy()

# Ensure 'Date' is in datetime format
filtered_data7['Date'] = pd.to_datetime(filtered_data7['Date'], format='%d/%m/%Y')

# Print the filtered dataset
print(filtered_data7)


           Date      Open      High       Low     Close
1419 2022-02-23  57632.94  57733.37  57109.24  57232.06
1420 2022-02-24  55418.45  55996.09  54383.20  54529.91
1421 2022-02-25  55321.72  56183.70  55299.28  55858.52
1422 2022-02-28  55329.46  56324.54  54833.50  56247.28
1423 2022-03-02  55629.30  55755.09  55020.10  55468.90
...         ...       ...       ...       ...       ...
1698 2023-04-12  60180.20  60437.64  60094.69  60392.77
1699 2023-04-13  60364.41  60486.91  60081.43  60431.00
1700 2023-04-17  60385.90  60407.86  59442.47  59910.75
1701 2023-04-18  59991.26  60113.47  59579.30  59727.01
1702 2023-04-19  59745.89  59745.89  59452.72  59567.80

[284 rows x 5 columns]


Imputation of Closing values for weekends and holidays

In [217]:
# Ensure 'Date' is in datetime format and set as index
filtered_data7['Date'] = pd.to_datetime(filtered_data7['Date'])
filtered_data7.set_index('Date', inplace=True)

# Create a full range of dates between the min and max dates in the dataset
full_range7 = pd.date_range(start=filtered_data7.index.min(), end=filtered_data7.index.max())

# Reindex the data to fill any missing dates
filtered_data7 = filtered_data7.reindex(full_range7)

# Forward fill missing 'Close' values
filtered_data7['Close'] = filtered_data7['Close'].ffill()

# Reset the index and rename it back to 'Date'
filtered_data7.reset_index(inplace=True)
filtered_data7.rename(columns={'index': 'Date'}, inplace=True)

# Print the updated filtered dataset
print(filtered_data7)


          Date      Open      High       Low     Close
0   2022-02-23  57632.94  57733.37  57109.24  57232.06
1   2022-02-24  55418.45  55996.09  54383.20  54529.91
2   2022-02-25  55321.72  56183.70  55299.28  55858.52
3   2022-02-26       NaN       NaN       NaN  55858.52
4   2022-02-27       NaN       NaN       NaN  55858.52
..         ...       ...       ...       ...       ...
416 2023-04-15       NaN       NaN       NaN  60431.00
417 2023-04-16       NaN       NaN       NaN  60431.00
418 2023-04-17  60385.90  60407.86  59442.47  59910.75
419 2023-04-18  59991.26  60113.47  59579.30  59727.01
420 2023-04-19  59745.89  59745.89  59452.72  59567.80

[421 rows x 5 columns]


Required Libraries

In [218]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input
import plotly.graph_objs as go
from sklearn.linear_model import LinearRegression

In [219]:
def rmse7(y_true7, y_pred7):
    return np.sqrt(mean_squared_error(y_true7, y_pred7))


Scale Data 

In [220]:
test['Date'] = pd.to_datetime(test['Date'], format='%d/%m/%Y')
filtered_data7['Date'] = pd.to_datetime(filtered_data7['Date'])

scaler7 = MinMaxScaler(feature_range=(0, 1))
scaled_data7 = scaler7.fit_transform(filtered_data7[['Close']].values)


Create sequences for LSTM input

In [221]:
def create_sequences7(data, seq_length7):
    X7, y7 = [], []
    for i in range(len(data) - seq_length7):
        X7.append(data[i:i + seq_length7])
        y7.append(data[i + seq_length7])
    return np.array(X7), np.array(y7)

# Hyperparameters
seq_length7 = 50
X7, y7 = create_sequences7(scaled_data7, seq_length7)


Split the data into training and test sets & Build and compile the LSTM model

In [222]:
split_index7 = int(0.8 * len(X7))
X_train7, X_test7 = X7[:split_index7], X7[split_index7:]
y_train7, y_test7 = y7[:split_index7], y7[split_index7:]

X_train7 = np.reshape(X_train7, (X_train7.shape[0], X_train7.shape[1], 1))
X_test7 = np.reshape(X_test7, (X_test7.shape[0], X_test7.shape[1], 1))

model7 = Sequential()
model7.add(Input(shape=(seq_length7, 1)))
model7.add(LSTM(50, return_sequences=True))
model7.add(LSTM(50))
model7.add(Dense(1))

# Compile the model
model7.compile(optimizer='adam', loss='mean_squared_error')


Train the model 

In [223]:
history7 = model7.fit(X_train7, y_train7, epochs=20, batch_size=32, verbose=1)
predictions7 = model7.predict(X_test7)
predictions7 = scaler7.inverse_transform(predictions7)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual7 = scaler7.inverse_transform(y_test7.reshape(-1, 1))

# Compute evaluation metrics
mape7 = mean_absolute_percentage_error(y_test_actual7, predictions7) * 100
rmse_value7 = rmse7(y_test_actual7, predictions7)


Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.1816
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0229
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0157
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0125
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0091
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0079
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0078
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0076
Epoch 9/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0070
Epoch 10/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0068
Epoch 11/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0065
Epoch 12/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0056
Epoch 13/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0053
Epoch 14/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0059
Epoch 15/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0052
Epoc

Make predictions and inverse scaling

In [224]:
predictions7 = model7.predict(X_test7)
predictions7 = scaler7.inverse_transform(predictions7)  # Inverse scaling

# Inverse scale the true values as well
y_test_actual7 = scaler7.inverse_transform(y_test7.reshape(-1, 1))

# Compute evaluation metrics
mape7 = mean_absolute_percentage_error(y_test_actual7, predictions7) * 100
rmse_value7 = rmse7(y_test_actual7, predictions7)

print(mape7)
print(rmse_value7)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
0.9435775346320834
700.8637265053845


Moving average and linear trend line

In [225]:
# Moving Average (5 Days)
filtered_data7['Moving_Average'] = filtered_data7['Close'].rolling(window=5).mean()

# Linear Trend Line
X_linear7 = filtered_data7['Date'].map(pd.Timestamp.toordinal).values.reshape(-1, 1)
linear_model7 = LinearRegression()
linear_model7.fit(X_linear7, filtered_data7['Close'])
filtered_data7['Linear_Trend'] = linear_model7.predict(X_linear7)


Graph: Show entire dataset (original + prediction + moving average + trend)

In [226]:
# Plot the results for the first set of interest rate cut events

# Original Close Values
trace_original7 = go.Scatter(
    x=filtered_data7['Date'], 
    y=filtered_data7['Close'], 
    mode='lines', 
    name='Original Close Values',
    line=dict(color='grey')
)

# LSTM Predictions
trace_lstm7 = go.Scatter(
    x=filtered_data7['Date'][split_index7+seq_length7:], 
    y=predictions7.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM)',
    line=dict(color='blue')
)

# Moving Average (5 Days)
trace_moving_avg7 = go.Scatter(
    x=filtered_data7['Date'], 
    y=filtered_data7['Moving_Average'], 
    mode='lines', 
    name='Moving Average (5 Days)',
    line=dict(color='green', dash='dot')
)

# Linear Trend Line
trace_linear_trend7 = go.Scatter(
    x=filtered_data7['Date'], 
    y=filtered_data7['Linear_Trend'], 
    mode='lines', 
    name='Linear Trend',
    line=dict(color='red', dash='dash')
)

# Vertical Lines for interest rate hikes
vertical_lines7 = [
    go.Scatter(
        x=[pd.to_datetime('2022-05-04'), pd.to_datetime('2022-05-04')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='May 4, 2022: Hike to 4.40%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2022-06-08'), pd.to_datetime('2022-06-08')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Jun 8, 2022: Hike to 4.90%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2022-08-05'), pd.to_datetime('2022-08-05')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Aug 5, 2022: Hike to 5.40%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2022-09-30'), pd.to_datetime('2022-09-30')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Sep 30, 2022: Hike to 5.90%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2022-12-07'), pd.to_datetime('2022-12-07')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Dec 7, 2022: Hike to 6.25%'
    ),
    go.Scatter(
        x=[pd.to_datetime('2023-02-08'), pd.to_datetime('2023-02-08')],
        y=[filtered_data7['Close'].min(), filtered_data7['Close'].max()],
        mode='lines',
        line=dict(color='black', dash='dash'),
        name='Feb 8, 2023: Hike to 6.50%'
    )
]

# Combine all traces for the first graph
layout7 = go.Layout(
    title=f"Entire Dataset with LSTM Forecast, Moving Average, and Linear Trend (MAPE: {mape7:.2f}%, RMSE: {rmse_value7:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig7 = go.Figure(data=[trace_original7, trace_lstm7, trace_moving_avg7, trace_linear_trend7] + vertical_lines7, layout=layout7)
fig7.show()


Graph: Show zoomed-in view starting from predictions

In [227]:
# Second graph: Zoom in on prediction period only

# Zoomed-in Original Close Values
trace_original_zoomed7 = go.Scatter(
    x=filtered_data7['Date'][split_index7+seq_length7:], 
    y=y_test_actual7.flatten(), 
    mode='lines', 
    name='Original Close Values (Zoomed)',
    line=dict(color='grey')
)

# Zoomed-in LSTM Predictions
trace_lstm_zoomed7 = go.Scatter(
    x=filtered_data7['Date'][split_index7+seq_length7:], 
    y=predictions7.flatten(), 
    mode='lines', 
    name='Forecasted Close Values (LSTM, Zoomed)',
    line=dict(color='blue')
)

# Combine traces for zoomed-in graph
layout_zoomed7 = go.Layout(
    title=f"Zoomed View: LSTM Forecast vs Original Close Values (MAPE: {mape7:.2f}%, RMSE: {rmse_value7:.2f})",
    xaxis_title="Date",
    yaxis_title="Close Values",
    font=dict(color="black"),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True)
)

fig_zoomed7 = go.Figure(data=[trace_original_zoomed7, trace_lstm_zoomed7], layout=layout_zoomed7)
fig_zoomed7.show()
